# One-epoch GPU profile for Julia full-basis NN training

This notebook profiles one non-Monte-Carlo GPU training epoch for the Julia neural-network ansatz. It mirrors the `trainNN` GPU path in `tJ_julia/tJNetRC.jl`, but times the main pieces separately:

- CPU Hamiltonian assembly and GPU transfer
- full-basis NN input construction
- NN forward pass
- sparse variational loss for a fixed `psi`
- Zygote gradient pass through forward + variational loss
- Flux optimizer update

The default configuration is the 18-site, 2-hole sector with 8 spin-up and 8 spin-down electrons.

In [15]:
using LinearAlgebra
using Random
using SparseArrays
using Statistics
using Dates
using Printf

using Flux
using Zygote
import CUDA

function find_tj_dir()
    candidates = unique([pwd(), joinpath(pwd(), "tJ_julia"), @__DIR__])
    for dir in candidates
        if isfile(joinpath(dir, "tJ.jl")) && isfile(joinpath(dir, "tJNetRC.jl"))
            return dir
        end
    end
    error("Could not find tJ.jl and tJNetRC.jl. Start Julia from the project root or tJ_julia/.")
end

const TJ_DIR = find_tj_dir()
include(joinpath(TJ_DIR, "tJ.jl"))
include(joinpath(TJ_DIR, "tJNetRC.jl"))

using .TJModels
using .TJNetRC

TJNetRC.gpu_available() || error("No CUDA GPU detected by TJNetRC.gpu_available(). Run this on gpu_test.")
CUDA.functional() || error("CUDA.jl is not functional in this session.")
CUDA.allowscalar(false)

const RESULTS_DIR = joinpath(TJ_DIR, "profile_results")
mkpath(RESULTS_DIR)
const RESULTS_PATH = joinpath(RESULTS_DIR, "gpu_one_epoch_profile.tsv")

(TJ_DIR=TJ_DIR, RESULTS_PATH=RESULTS_PATH, cuda_device=CUDA.name(CUDA.device()))

(TJ_DIR = "/n/holylabs/kaxiras_lab/axzhu/projects/tJ_julia", RESULTS_PATH = "/n/holylabs/kaxiras_lab/axzhu/projects/tJ_julia/profile_results/gpu_one_epoch_profile.tsv", cuda_device = "NVIDIA A100-SXM4-40GB MIG 3g.20gb")

In [16]:
const SITES = 18
const HOLES = 2
const NUP = 8
const NDOWN = 8
const LATTICE_SPEC = (m=3, n=0, mp=0, np=3)

const T_VALUE = 1.0
const J_OVER_T = 0.3
const TCOEF = -T_VALUE
const JCOEF = J_OVER_T * T_VALUE

const N_NEURONS = 128
const DROPOUT = 0.0
const NN_LEARNING_RATE = 1.0e-3
const RNG_SEED = 12345
const FLOAT_TYPE = Float32
const TRUNC = nothing

# Run one unrecorded warmup epoch first so the profiled epoch is not dominated by compilation/kernel startup.
const RUN_WARMUP_EPOCH = true

# Multi-epoch blocks help catch allocator/GC/cache effects that a single warmed epoch can hide.
const PROFILE_BLOCK_EPOCH_COUNTS = (10, 100)

# Keep this aligned with the trainNN params you want to profile.
const PARAMS = Dict{String, Any}(
    "device" => "gpu",
    "n_neurons" => N_NEURONS,
    "dropout" => DROPOUT,
    "nn_learning_rate" => NN_LEARNING_RATE,
    "optimizer" => "adam",
    "epochs" => 1,
    "loss_diff" => 1.0e-4,
    "stagnation" => 20,
    "verbose" => false,
    "model_path" => nothing,
    "load_from" => nothing,
    "exploit_symmetry" => false,
)

const DEVICE = TJNetRC._resolve_device(PARAMS; context="profile trainNN")
const EXPLOIT_SYMMETRY = Bool(TJNetRC._param(PARAMS, "exploit_symmetry", false))

(SITES=SITES, HOLES=HOLES, NUP=NUP, NDOWN=NDOWN, J_OVER_T=J_OVER_T, N_NEURONS=N_NEURONS, device=TJNetRC._device_label(DEVICE), exploit_symmetry=EXPLOIT_SYMMETRY)

(SITES = 18, HOLES = 2, NUP = 8, NDOWN = 8, J_OVER_T = 0.3, N_NEURONS = 128, device = "gpu", exploit_symmetry = false)

In [17]:
function timed_cuda(f)
    result = nothing
    elapsed = @elapsed begin
        result = f()
        CUDA.synchronize()
    end
    return result, elapsed
end

function scalar_float64(x)
    x isa Number && return Float64(x)
    return Float64(only(Array(x)))
end

function tsv_cell(x)
    (x === missing || x === nothing) && return ""
    return replace(string(x), '\t' => ' ', '\n' => ' ')
end

const PROFILE_COLUMNS = [
    "timestamp", "stage", "seconds", "sites", "holes", "nup", "ndown", "dim", "nnz",
    "J_over_t", "n_neurons", "loss", "cuda_device", "H_type", "input_type", "notes",
]

const PROFILE_DIM = Ref{Int}(0)
const PROFILE_NNZ = Ref{Int}(0)
const PROFILE_H_TYPE = Ref{String}("")
const PROFILE_INPUT_TYPE = Ref{String}("")

function append_profile_rows!(path, rows)
    isempty(rows) && return path
    new_file = !isfile(path) || filesize(path) == 0
    open(path, "a") do io
        new_file && println(io, join(PROFILE_COLUMNS, "\t"))
        for row in rows
            println(io, join((tsv_cell(get(row, col, missing)) for col in PROFILE_COLUMNS), "\t"))
        end
    end
    return path
end

function base_profile_row(stage, seconds; loss=missing, notes="")
    return Dict{String, Any}(
        "timestamp" => Dates.format(now(), dateformat"yyyy-mm-ddTHH:MM:SS"),
        "stage" => stage,
        "seconds" => seconds,
        "sites" => SITES,
        "holes" => HOLES,
        "nup" => NUP,
        "ndown" => NDOWN,
        "dim" => PROFILE_DIM[],
        "nnz" => PROFILE_NNZ[],
        "J_over_t" => J_OVER_T,
        "n_neurons" => Int(TJNetRC._param(PARAMS, "n_neurons")),
        "loss" => loss,
        "cuda_device" => CUDA.name(CUDA.device()),
        "H_type" => PROFILE_H_TYPE[],
        "input_type" => PROFILE_INPUT_TYPE[],
        "notes" => notes,
    )
end

base_profile_row (generic function with 1 method)

## Build model, Hamiltonian, inputs, and network

These setup timings are not per-epoch training costs, but they are useful for separating one-time preparation from the actual training epoch.

In [18]:
Random.seed!(RNG_SEED)

setup_rows = Dict{String, Any}[]

model = nothing
model_seconds = @elapsed begin
    model = TJModels.TJ(NUP, NDOWN, LATTICE_SPEC.m, LATTICE_SPEC.n, LATTICE_SPEC.mp, LATTICE_SPEC.np)
end
@assert model.M == SITES
@assert model.h == HOLES
neighbours = TJModels.kneighbours(model, 1)
@assert length(neighbours) == 3 * model.M ÷ 2

input_data = nothing
input_seconds = @elapsed begin
    input_data = TJNetRC.NN_inputs(TCOEF, JCOEF, model; trunc=TRUNC, device=DEVICE)
    CUDA.synchronize()
end
dim = size(input_data, 2)

terms = nothing
mats = nothing
Ht_work = nothing
HJ_work = nothing
H_cpu = nothing
H_cpu_type = ""
assembly_seconds = @elapsed begin
    terms = TJModels.hamiltonian_by_degree(model; degrees=(1,))
    mats = TJModels.assemble_by_degree(model, terms)
    Ht_work = TJNetRC._slice_hamiltonian(mats.t[1], dim)
    HJ_work = TJNetRC._slice_hamiltonian(mats.J[1], dim)
    H_cpu = TCOEF .* Ht_work .+ JCOEF .* HJ_work
    if EXPLOIT_SYMMETRY
        H_cpu = TJNetRC._symmetry_reduce(H_cpu, dim; phase=1)
        input_data = TJNetRC._symmetry_inputs(input_data, dim)
    end
end
PROFILE_DIM[] = size(input_data, 2)
PROFILE_NNZ[] = nnz(H_cpu)
H_cpu_type = string(typeof(H_cpu))
PROFILE_INPUT_TYPE[] = string(typeof(input_data))

H_gpu = nothing
H_transfer_seconds = @elapsed begin
    H_gpu = TJNetRC.processH(H_cpu; device=DEVICE)
    CUDA.synchronize()
end
PROFILE_H_TYPE[] = string(typeof(H_gpu))

optimizer_name = TJNetRC._full_optimizer_name(PARAMS, DEVICE)
if DEVICE.name == :gpu && optimizer_name == "lbfgs"
    @warn "Optim.LBFGS is CPU-oriented in this training path; using Adam for GPU full-basis training."
    optimizer_name = "adam"
end
learning_rate = Float64(TJNetRC._param(PARAMS, "nn_learning_rate", 1.0e-3))

function initialize_training_state()
    net = TJNetRC._to_device(
        TJNetRC._load_or_initialize_network(
            PARAMS,
            model.M,
            Int(TJNetRC._param(PARAMS, "n_neurons")),
            Float64(TJNetRC._param(PARAMS, "dropout", 0.0)),
            RNG_SEED,
        ),
        DEVICE,
    )
    state = Flux.setup(TJNetRC._flux_optimizer(optimizer_name, learning_rate), net)
    CUDA.synchronize()
    return net, state
end

neuralnet = nothing
opt_state = nothing
network_seconds = @elapsed begin
    neuralnet, opt_state = initialize_training_state()
end

push!(setup_rows, base_profile_row("model_basis", model_seconds; notes="CPU model and basis construction"))
push!(setup_rows, base_profile_row("input_to_gpu", input_seconds; notes="NN_inputs including GPU transfer"))
push!(setup_rows, base_profile_row("hamiltonian_assembly_cpu", assembly_seconds; notes="CPU nearest-neighbor H assembly matching trainNN H=t*Ht+J*HJ"))
push!(setup_rows, base_profile_row("hamiltonian_to_gpu", H_transfer_seconds; notes="processH GPU sparse conversion"))
push!(setup_rows, base_profile_row("network_to_gpu_optimizer_setup", network_seconds; notes="_load_or_initialize_network, _to_device, Flux.setup"))

setup_summary = (
    dim=PROFILE_DIM[],
    nnz=PROFILE_NNZ[],
    H_cpu_type=H_cpu_type,
    H_gpu_type=typeof(H_gpu),
    input_size=size(input_data),
    input_type=typeof(input_data),
    optimizer=optimizer_name,
    device=TJNetRC._device_label(DEVICE),
    setup_rows=setup_rows,
)
setup_summary

(dim = 1969110, nnz = 35328150, H_cpu_type = "SparseMatrixCSC{Float64, Int64}", H_gpu_type = CUDA.CUSPARSE.CuSparseMatrixCSC{Float32, Int32}, input_size = (20, 1969110), input_type = CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}, optimizer = "adam", device = "gpu", setup_rows = Dict{String, Any}[Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" => 8, "stage" => "model_basis", "seconds" => 3.414248532, "loss" => missing, "nup" => 8, "holes" => 2, "nnz" => 35328150, "H_type" => "CUDA.CUSPARSE.CuSparseMatrixCSC{Float32, Int32}"…), Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" => 8, "stage" => "input_to_gpu", "seconds" => 0.45747111, "loss" => missing, "nup" => 8, "holes" => 2, "nnz" => 35328150, "H_type" => "CUDA.CUSPARSE.CuSparseMatrixCSC{Float32, Int32}"…), Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" => 8, "stage" => "hamiltonian_assembly_cpu"

In [19]:
function profile_forward_psi(net)
    scale = EXPLOIT_SYMMETRY ? inv(sqrt(eltype(input_data)(2))) : one(eltype(input_data))
    return vec(net(input_data)) .* scale
end

function profile_loss(net)
    return TJNetRC._variational_loss(net, H_gpu, input_data; exploit_symmetry=EXPLOIT_SYMMETRY)
end

function gradient_step_only(net)
    loss_ref = Ref{Any}()
    grads = Zygote.gradient(net) do current_net
        loss = profile_loss(current_net)
        Zygote.ignore_derivatives() do
            loss_ref[] = loss
        end
        return loss
    end
    grads[1] === nothing && error("Gradient was nothing")
    return grads[1], loss_ref[]
end

function trainnn_bookkeeping!(loss_history, prev_loss, stagnation_counter, loss_value)
    current_loss = TJNetRC._scalar_float64(loss_value)
    push!(loss_history, current_loss)

    loss_diff = Float64(TJNetRC._param(PARAMS, "loss_diff", 1.0e-4))
    if abs(current_loss - prev_loss[]) < loss_diff
        stagnation_counter[] += 1
    else
        stagnation_counter[] = 0
    end
    prev_loss[] = current_loss
    return current_loss
end

function update_and_bookkeeping!(opt_state, net, grads, loss_value, loss_history, prev_loss, stagnation_counter)
    Flux.update!(opt_state, net, grads)
    return trainnn_bookkeeping!(loss_history, prev_loss, stagnation_counter, loss_value)
end

function trainnn_epoch_exact!(net, opt_state, loss_history, prev_loss, stagnation_counter)
    grads, loss_value = gradient_step_only(net)
    return update_and_bookkeeping!(opt_state, net, grads, loss_value, loss_history, prev_loss, stagnation_counter)
end

function trainnn_epochs_exact!(net, opt_state, requested_epochs::Integer)
    loss_history = Float64[]
    prev_loss = Ref(Inf)
    stagnation_counter = Ref(0)
    stagnation = Int(TJNetRC._param(PARAMS, "stagnation", 20))
    current_loss = NaN

    for _ in 1:requested_epochs
        current_loss = trainnn_epoch_exact!(net, opt_state, loss_history, prev_loss, stagnation_counter)
        stagnation_counter[] >= stagnation && break
    end

    return (loss=current_loss, epochs=length(loss_history), stopped=length(loss_history) < requested_epochs)
end

trainnn_epochs_exact! (generic function with 1 method)

## Optional warmup epoch

The warmup is not included in the profile rows. It compiles kernels and AD code before the measured epoch.

In [20]:
warmup_seconds = missing
warmup_loss = missing

if RUN_WARMUP_EPOCH
    warmup_net, warmup_opt_state = initialize_training_state()
    warmup_history = Float64[]
    warmup_prev_loss = Ref(Inf)
    warmup_stagnation_counter = Ref(0)
    warmup_loss, warmup_seconds = timed_cuda(() -> trainnn_epoch_exact!(
        warmup_net,
        warmup_opt_state,
        warmup_history,
        warmup_prev_loss,
        warmup_stagnation_counter,
    ))
    neuralnet, opt_state = initialize_training_state()
end

(RUN_WARMUP_EPOCH=RUN_WARMUP_EPOCH, warmup_seconds=warmup_seconds, warmup_loss=warmup_loss)

(RUN_WARMUP_EPOCH = true, warmup_seconds = 0.771992473, warmup_loss = -6.0053558349609375)

## Profile one epoch

`trainnn_epoch_exact` mirrors one `_optimise_full_flux!` epoch end-to-end, with no synchronization between gradient and update. The `forward_only`, `variational_given_psi`, and decomposed rows are diagnostic subpieces and include extra synchronization boundaries.

In [21]:
profile_rows = copy(setup_rows)

exact_neuralnet, exact_opt_state = initialize_training_state()
exact_loss_history = Float64[]
exact_prev_loss = Ref(Inf)
exact_stagnation_counter = Ref(0)
exact_loss, exact_epoch_seconds = timed_cuda(() -> trainnn_epoch_exact!(
    exact_neuralnet,
    exact_opt_state,
    exact_loss_history,
    exact_prev_loss,
    exact_stagnation_counter,
))
push!(profile_rows, base_profile_row(
    "trainnn_epoch_exact",
    exact_epoch_seconds;
    loss=exact_loss,
    notes="One _optimise_full_flux! epoch: gradient, Flux.update!, scalar loss/history/stagnation bookkeeping",
))

for block_epochs in PROFILE_BLOCK_EPOCH_COUNTS
    block_net, block_opt_state = initialize_training_state()
    block_result, block_seconds = timed_cuda(() -> trainnn_epochs_exact!(block_net, block_opt_state, block_epochs))
    per_epoch = block_result.epochs == 0 ? NaN : block_seconds / block_result.epochs
    push!(profile_rows, base_profile_row(
        "trainnn_$(block_epochs)_epoch_exact_block",
        block_seconds;
        loss=block_result.loss,
        notes="$(block_result.epochs) epochs, per_epoch=$(round(per_epoch; digits=6))s, stopped=$(block_result.stopped); mirrors _optimise_full_flux! loop",
    ))
end

psi_forward, forward_seconds = timed_cuda(() -> profile_forward_psi(neuralnet))
push!(profile_rows, base_profile_row("forward_only", forward_seconds; notes="profile_forward_psi(net), no loss/backward"))

loss_given_psi, variational_seconds = timed_cuda(() -> TJNetRC.variational(H_gpu, psi_forward))
push!(profile_rows, base_profile_row(
    "variational_given_psi",
    variational_seconds;
    loss=TJNetRC._scalar_float64(loss_given_psi),
    notes="sparse H*psi plus Rayleigh quotient, no NN forward/backward",
))

gradient_result, gradient_seconds = timed_cuda(() -> gradient_step_only(neuralnet))
grads, profiled_loss_device = gradient_result

_, update_seconds = timed_cuda(() -> Flux.update!(opt_state, neuralnet, grads))

loss_history = Float64[]
prev_loss = Ref(Inf)
stagnation_counter = Ref(0)
profiled_loss, bookkeeping_seconds = timed_cuda(() -> trainnn_bookkeeping!(
    loss_history,
    prev_loss,
    stagnation_counter,
    profiled_loss_device,
))

push!(profile_rows, base_profile_row(
    "gradient_forward_loss_backward",
    gradient_seconds;
    loss=profiled_loss,
    notes="Zygote.gradient over _variational_loss; includes NN forward, sparse loss, and backward",
))
push!(profile_rows, base_profile_row("optimizer_update", update_seconds; loss=profiled_loss, notes="Flux.update! with precomputed grads"))
push!(profile_rows, base_profile_row("trainnn_bookkeeping", bookkeeping_seconds; loss=profiled_loss, notes="_scalar_float64, loss_history push!, loss_diff/stagnation update"))
grads = nothing
gradient_result = nothing

push!(profile_rows, base_profile_row(
    "profiled_epoch_total",
    gradient_seconds + update_seconds + bookkeeping_seconds;
    loss=profiled_loss,
    notes="decomposed total; includes synchronization between subpieces, unlike trainnn_epoch_exact",
))

profile_rows

14-element Vector{Dict{String, Any}}:
 Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" => 8, "stage" => "model_basis", "seconds" => 3.414248532, "loss" => missing, "nup" => 8, "holes" => 2, "nnz" => 35328150, "H_type" => "CUDA.CUSPARSE.CuSparseMatrixCSC{Float32, Int32}"…)
 Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" => 8, "stage" => "input_to_gpu", "seconds" => 0.45747111, "loss" => missing, "nup" => 8, "holes" => 2, "nnz" => 35328150, "H_type" => "CUDA.CUSPARSE.CuSparseMatrixCSC{Float32, Int32}"…)
 Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" => 8, "stage" => "hamiltonian_assembly_cpu", "seconds" => 40.487064253, "loss" => missing, "nup" => 8, "holes" => 2, "nnz" => 35328150, "H_type" => "CUDA.CUSPARSE.CuSparseMatrixCSC{Float32, Int32}"…)
 Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" =>

In [22]:
append_profile_rows!(RESULTS_PATH, profile_rows)

(RESULTS_PATH=RESULTS_PATH, rows=profile_rows)

(RESULTS_PATH = "/n/holylabs/kaxiras_lab/axzhu/projects/tJ_julia/profile_results/gpu_one_epoch_profile.tsv", rows = Dict{String, Any}[Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" => 8, "stage" => "model_basis", "seconds" => 3.414248532, "loss" => missing, "nup" => 8, "holes" => 2, "nnz" => 35328150, "H_type" => "CUDA.CUSPARSE.CuSparseMatrixCSC{Float32, Int32}"…), Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" => 8, "stage" => "input_to_gpu", "seconds" => 0.45747111, "loss" => missing, "nup" => 8, "holes" => 2, "nnz" => 35328150, "H_type" => "CUDA.CUSPARSE.CuSparseMatrixCSC{Float32, Int32}"…), Dict("n_neurons" => 128, "input_type" => "CUDA.CuArray{Float32, 2, CUDA.DeviceMemory}", "ndown" => 8, "stage" => "hamiltonian_assembly_cpu", "seconds" => 40.487064253, "loss" => missing, "nup" => 8, "holes" => 2, "nnz" => 35328150, "H_type" => "CUDA.CUSPARSE.CuSparseMatrixCSC{Float32, Int32}"…), Dic

## Optional CUDA profiler hook

The cell below is intentionally commented out. Uncomment it if you want a CUDA-level profile of a single full epoch in an interactive session.

In [ ]:
# CUDA.@profile begin
#     trainnn_epoch_exact!(neuralnet, opt_state, Float64[], Ref(Inf), Ref(0))
#     CUDA.synchronize()
# end